# Supplemental alternate workflow — not the manuscript analysis

This notebook is retained as a user-supplied reference. It is not part of the canonical workflow described in the paper and has not been validated against the paper's methods or results. It uses a different Approach 1/2 assignment and legacy `$path` placeholders. Notebook execution outputs were removed from this project copy because they contain stale or discrepant results. The original files remain in the user's Downloads folder.


Cell 1 | markdown
# scRNA-seq Pipeline — Step 1: Preprocessing

> **Goal:** Reproduce, in one place, everything the downstream re-typing/marker notebooks assume already
> exists: the raw-to-annotated preprocessing pipeline from `process_pcluster_yg.ipynb`, copied verbatim
> from its beginning through **Step 5 — Cell Type Annotation with CellTypist**.

**This notebook is a straight copy of `process_pcluster_yg.ipynb`'s first 21 cells — no logic changed.**
Run it top to bottom before running **`scRNAseq_2.1_CellReTypeSupervised.ipynb`**, which consumes its
final output:

```
$path/5.pre_hvg_annotation/merged_celltypist_immune_low.h5ad
```

## Pipeline position

**`scRNAseq_1_Preprocess.ipynb`** (this notebook) → `scRNAseq_2.1_CellReTypeSupervised.ipynb` →
`scRNAseq_2.2_CellReTypeUnsupervised.ipynb` → `scRNAseq_3_GeneMarker.ipynb`

## Steps included

| Step | What |
|---|---|
| Prereq | nf-lly-scrnaseq pipeline inputs, metadata prep |
| 1.1 | QC Raw — per-sample metrics |
| 1.2 | Scanpy Scrublet — doublet detection |
| 1.3 | Doublet removal — merge and filter |
| 2.1 | QC filter — mitochondrial / gene / cell thresholds |
| 2.2 | Finalize QC AnnDatas |
| 3.1 | Unify — ADATA_UNIFY |
| 3.2 | Unify — UPSET_GENES |
| 4 | Combine — ADATA_MERGE |
| 5 | Cell-type annotation with CellTypist (final output for downstream notebooks) |

Cell 2 | markdown
## Prequest: Pipeline Used Before Running Secondary Analsysis
The input data was generated using the nf-lly-scrnaseq pipeline with the following command:

```bash
nextflow run ~/scripts/nf-lly-snrnaseq/main.nf \
  -profile pcluster \
  --sample_sheet test_nf-lly-snRNA.csv \
  --fasta ~/references/10xh38/refdata-gex-GRCh38-2024-A/fasta/genome.fa \
  --gtf ~/references/10xh38/refdata-gex-GRCh38-2024-A/genes/genes.gtf \
  --protocol auto \
  --aligner cellranger \
  --outdir output/
```

### Input Data
This analysis uses processed H5AD files from the pipeline's `mtx_conversions` step:
- **Filtered matrix:** `*_emptydrops_filter_matrix.h5ad` (filtered cells)
- **Raw matrix:** `*_raw_matrix.h5ad` (unfiltered matrix)

#### Example Input Paths
For sample `SRR14575500`:
```
output/cellranger/mtx_conversions/SRR14575500/SRR14575500_emptydrops_filter_matrix.h5ad
output/cellranger/mtx_conversions/SRR14575500/SRR14575500_raw_matrix.h5ad
```

Cell 3 | markdown
## Prepare h5ad by adding the metadata before running next step

In [ ]:
# Cell 4 | code
import scanpy as sc
from pathlib import Path

# Directory containing one folder per sample (e.g., SRR14575500/)
BASE_DIR = Path("$path/cellranger/mtx_conversions")

# Output directory for metadata-added h5ad files (kept separate from originals)
OUT_DIR = Path("$path/meta_added_h5ad")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Fill in metadata for each sample folder.
# Keys must match folder names under BASE_DIR.
sample_metadata = {
    "SRR14575500": {"treatment": "ctr", "batch": "batch1"},
    "SRR14575504": {"treatment": "pre-treatment", "batch": "batch1"},
    "SRR14575508": {"treatment": "post-treatment", "batch": "batch1"},
}

required_fields = {"treatment", "batch"}
missing_samples = [s for s in sample_metadata if not (BASE_DIR / s).is_dir()]
if missing_samples:
    raise FileNotFoundError(f"These sample folders are missing under {BASE_DIR}: {missing_samples}")


def update_and_save_h5ad(input_path: Path, output_path: Path, meta: dict):
    adata = sc.read_h5ad(input_path)
    for field in required_fields:
        adata.obs[field] = meta[field]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(output_path)


updated = []
skipped = []

for sample_dir in sorted([p for p in BASE_DIR.iterdir() if p.is_dir()]):
    sample = sample_dir.name

    # Ignore directories you have not explicitly annotated.
    if sample not in sample_metadata:
        skipped.append(sample)
        continue

    meta = sample_metadata[sample]
    if not required_fields.issubset(meta.keys()):
        missing_fields = required_fields - set(meta.keys())
        raise ValueError(f"Metadata for {sample} is missing fields: {missing_fields}")

    filtered_files = sorted(sample_dir.glob("*emptydrops_filter_matrix.h5ad"))
    raw_files = sorted(sample_dir.glob("*raw_matrix.h5ad"))

    if len(filtered_files) != 1:
        raise FileNotFoundError(
            f"Expected exactly 1 filtered h5ad in {sample_dir}, found {len(filtered_files)}: {filtered_files}"
        )
    if len(raw_files) != 1:
        raise FileNotFoundError(
            f"Expected exactly 1 unfiltered h5ad in {sample_dir}, found {len(raw_files)}: {raw_files}"
        )

    out_sample_dir = OUT_DIR / sample
    file_pairs = [
        (filtered_files[0], out_sample_dir / filtered_files[0].name, "filtered"),
        (raw_files[0], out_sample_dir / raw_files[0].name, "unfiltered"),
    ]

    for in_h5ad, out_h5ad, matrix_type in file_pairs:
        update_and_save_h5ad(in_h5ad, out_h5ad, meta)
        updated.append((sample, matrix_type, str(in_h5ad), str(out_h5ad)))

print(f"Wrote {len(updated)} metadata-added files across {len(sample_metadata)} samples.")
print(f"Output root: {OUT_DIR}")
if skipped:
    print("Skipped folders not in sample_metadata:", skipped)

for sample, matrix_type, in_h5ad, out_h5ad in updated:
    print(f"[{sample}] {matrix_type}: {in_h5ad} -> {out_h5ad}")

Cell 5 | markdown
### Step 1.1 QC Raw — Per-Sample Metrics

In [ ]:
# Cell 6 | code
import os
import glob
import scanpy as sc
import matplotlib.pyplot as plt

mtx_dir = "$path/meta_added_h5ad"
base_out_dir = "$path/1.doublet_detection/1.1_qcraw"

h5ad_files = sorted(glob.glob(os.path.join(mtx_dir, "*", "*_emptydrops_filter_matrix.h5ad")))
print(f"Found {len(h5ad_files)} sample(s)")

for h5ad_path in h5ad_files:
    name = os.path.basename(h5ad_path).replace("_emptydrops_filter_matrix.h5ad", "")
    out_dir = os.path.join(base_out_dir, name)
    os.makedirs(out_dir, exist_ok=True)

    adata_filtered = sc.read_h5ad(h5ad_path)
    sc.pp.calculate_qc_metrics(adata_filtered, inplace=True)

    sc.pl.scatter(adata_filtered, x="total_counts", y="n_genes_by_counts", show=False)
    plt.savefig(os.path.join(out_dir, f"{name}_filtered_counts_vs_genes.png"), dpi=150, bbox_inches="tight")
    plt.close()

    with open(os.path.join(out_dir, f"{name}_convert_filtered_size.txt"), "w") as fh:
        fh.write(str(adata_filtered.n_obs))

    print(f"  {name}: {adata_filtered.n_obs} cells, {adata_filtered.n_vars} genes")

print("Step 1.1 QC Raw — done.")


Cell 7 | markdown
### Step 1.2 Scanpy Scrublet — Doublet Detection

In [ ]:
# Cell 8 | code
scrublet_dir = "$path/1.doublet_detection/1.2_scanpy_scrublet"
os.makedirs(scrublet_dir, exist_ok=True)

h5ad_files = sorted(glob.glob(os.path.join(mtx_dir, "*", "*_emptydrops_filter_matrix.h5ad")))

for h5ad_path in h5ad_files:
    name = os.path.basename(h5ad_path).replace("_emptydrops_filter_matrix.h5ad", "")
    prefix = f"{name}_scrublet"

    adata = sc.read_h5ad(h5ad_path)
    sc.pp.scrublet(adata)

    adata.obs["predicted_doublet"] = adata.obs["predicted_doublet"].astype(bool)
    df = adata.obs[["predicted_doublet"]].copy()
    df.columns = [prefix]
    df.to_pickle(os.path.join(scrublet_dir, f"{prefix}.pkl"))

    n_doublets = adata.obs["predicted_doublet"].sum()
    print(f"  {name}: {n_doublets}/{adata.n_obs} doublets ({n_doublets/adata.n_obs*100:.1f}%)")

    adata = adata[~adata.obs["predicted_doublet"]].copy()
    adata.write_h5ad(os.path.join(scrublet_dir, f"{prefix}.h5ad"))
    print(f"  {name}: {adata.n_obs} cells after doublet removal")

print("Step 1.2 Scanpy Scrublet — done.")


Cell 9 | markdown
### Step 1.3 Doublet Removal — Merge and Filter

In [ ]:
# Cell 10 | code
import anndata as ad
import pandas as pd
import warnings
import upsetplot
from upsetplot import from_memberships

base_doublet_removed_dir = "$path/1.doublet_detection/1.3_doublet_remove"
threshold = 1

def load_pred(path):
    if path.endswith(".pkl"):
        return pd.read_pickle(path)
    if path.endswith(".csv"):
        return pd.read_csv(path, index_col=0)
    raise ValueError(f"Unsupported file type: {path}")

h5ad_files = sorted(glob.glob(os.path.join(mtx_dir, "*", "*_emptydrops_filter_matrix.h5ad")))

for h5ad_path in h5ad_files:
    name = os.path.basename(h5ad_path).replace("_emptydrops_filter_matrix.h5ad", "")
    doublet_removed_dir = os.path.join(base_doublet_removed_dir, name)
    os.makedirs(doublet_removed_dir, exist_ok=True)

    adata = ad.read_h5ad(h5ad_path)
    scrublet_pkl = os.path.join(scrublet_dir, f"{name}_scrublet.pkl")
    predictions = pd.concat([load_pred(f) for f in scrublet_pkl.split()], axis=1)
    predictions = predictions.reindex(adata.obs_names)
    mask = predictions.sum(axis=1) >= threshold
    adata = adata[~mask, :]
    adata.write_h5ad(os.path.join(doublet_removed_dir, f"{name}_doublet_rm.h5ad"))

    contents = {col: predictions[col][predictions[col].astype(bool)].index.tolist() for col in predictions.columns}
    plot_data = from_memberships(contents.values(), data=None)
    fig = plt.figure()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        upsetplot.plot(plot_data, fig=fig)
    plt.savefig(os.path.join(doublet_removed_dir, f"{name}_predictions_mqc.png"), dpi=150, bbox_inches="tight")
    plt.close()

    with open(os.path.join(doublet_removed_dir, f"{name}_doublet_removed_size.txt"), "w") as f:
        f.write(str(adata.n_obs))

    print(f"  {name}: {mask.sum()} removed, {adata.n_obs} remaining")

print("Step 1.3 Doublet Removal — done.")


Cell 11 | markdown
### Step 2.1 QC Filter — Mitochondrial and Gene/Cell Thresholds

In [ ]:
# Cell 12 | code
from threadpoolctl import threadpool_limits

pct_mt_max   = 10
min_counts   = 0
min_genes    = 200
min_gene_counts = 0
min_cells    = 0
n_jobs       = 2

base_qc_filter_dir = "$path/2.qc_filter/2.1_qc_filter"

h5ad_files = sorted(glob.glob(os.path.join(base_doublet_removed_dir, "*", "*_doublet_rm.h5ad")))
print(f"Found {len(h5ad_files)} sample(s)")

for h5ad_path in h5ad_files:
    name = os.path.basename(h5ad_path).replace("_doublet_rm.h5ad", "")
    prefix = f"{name}_scanpy_filtered"
    qc_filter_dir = os.path.join(base_qc_filter_dir, name)
    os.makedirs(qc_filter_dir, exist_ok=True)

    threadpool_limits(n_jobs)
    sc.settings.n_jobs = n_jobs
    adata = sc.read_h5ad(h5ad_path)

    adata.var["mt"] = adata.var_names.str.lower().str.startswith("mt-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

    n_before = adata.n_obs
    adata = adata[adata.obs.pct_counts_mt < pct_mt_max, :].copy()
    sc.pp.filter_cells(adata, min_counts=min_counts)
    sc.pp.filter_genes(adata, min_counts=min_gene_counts)
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)

    adata.write_h5ad(os.path.join(qc_filter_dir, f"{prefix}.h5ad"))
    with open(os.path.join(qc_filter_dir, f"{prefix}_thresholded_size.txt"), "w") as fh:
        fh.write(str(adata.n_obs))

    print(f"  {name}: {n_before} -> {adata.n_obs} cells, {adata.n_vars} genes")

print("Step 2.1 QC Filter — done.")


Cell 13 | markdown
### Step 2.2 FINALIZE_QC_ANNDATAS (ADATA_EXTEND)

In [ ]:
# Cell 14 | code
import anndata as ad

finalize_qc_dir = "$path/2.qc_filter/2.2_finalize_qc"

h5ad_files = sorted(glob.glob(os.path.join(base_qc_filter_dir, "*", "*_scanpy_filtered.h5ad")))
print(f"Found {len(h5ad_files)} sample(s)")

for h5ad_path in h5ad_files:
    name = os.path.basename(h5ad_path).replace("_scanpy_filtered.h5ad", "")
    out_dir = os.path.join(finalize_qc_dir, name)
    os.makedirs(out_dir, exist_ok=True)

    adata = ad.read_h5ad(h5ad_path)
    # No additional obs/var/obsm/obsp/uns/layers in this run
    adata.write_h5ad(os.path.join(out_dir, f"{name}.h5ad"))
    adata.obs.to_csv(os.path.join(out_dir, f"{name}_metadata.csv"))
    print(f"  {name}: {adata.n_obs} cells, {adata.n_vars} genes")

print("Step 2.2 FINALIZE_QC_ANNDATAS — done.")


Cell 15 | markdown
### Step 3.1. UNIFY: ADATA_UNIFY

In [ ]:
# Cell 16 | code
import numpy as np
from scipy.sparse import csr_matrix

# Parameters (from nextflow.config / subworkflow defaults)
batch_col             = "batch"
label_col             = ""         # no cell-type labels
unknown_label         = "unknown"
symbol_col            = "index"    # gene symbols already in var_names
counts_layer          = "X"
duplicate_var_resolution = "sum"
aggregate_isoforms    = False

unify_dir = "$path/3.unify"
os.makedirs(unify_dir, exist_ok=True)

def aggregate_duplicate_var(adata, aggr_fun=np.mean):
    retain_var = ~adata.var_names.duplicated(keep="first")
    duplicated_var = adata.var_names[adata.var_names.duplicated()].unique()
    if len(duplicated_var):
        for var in duplicated_var:
            mask = adata.var_names == var
            var_aggr = aggr_fun(adata.X[:, mask], axis=1)[:, np.newaxis]
            adata.X[:, mask] = np.repeat(var_aggr, np.sum(mask), axis=1)
        return adata[:, retain_var].copy()
    return adata

def to_Florent_case(s):
    corrected = s.lower().strip()
    if corrected in ["na", "nan", "null", "unknown"]:
        return "unknown"
    corrected = s.replace(" ", "_").replace("-", "_")
    corrected = "".join([c if c.isalnum() or c == "_" else "" for c in corrected])
    corrected = corrected.replace("__", "_")
    if corrected.endswith("s"):
        corrected = corrected[:-1]
    corrected = corrected.strip(" _")
    return corrected.capitalize() if corrected else "unknown"

h5ad_files = sorted(glob.glob(os.path.join(finalize_qc_dir, "*", "*.h5ad")))
unified_paths = {}

for h5ad_path in h5ad_files:
    name = os.path.basename(os.path.dirname(h5ad_path))  # sample folder name

    adata = ad.read_h5ad(h5ad_path)

    # Use raw counts from X
    if counts_layer != "X":
        adata.X = adata.layers[counts_layer]

    # Strip computed slots
    adata.obsm = {}
    adata.varm = {}
    adata.uns  = {}
    adata.layers = {}

    # Convert to float32 CSR
    adata.X = csr_matrix(adata.X.astype(np.float32))

    # Aggregate isoforms (disabled)
    if aggregate_isoforms:
        adata.var_names = adata.var_names.str.replace(r'\.\d+', '', regex=True)

    # Resolve duplicate var names
    if duplicate_var_resolution in ["mean", "sum", "max"]:
        adata = aggregate_duplicate_var(adata, aggr_fun=getattr(np, duplicate_var_resolution))
    elif duplicate_var_resolution == "make_unique":
        adata.var_names_make_unique()

    # Unique obs names, prefixed with sample id
    adata.obs_names_make_unique()
    adata.obs_names = name + "_" + adata.obs_names

    # Batch column (already "batch" in obs)
    if batch_col not in adata.obs:
        adata.obs[batch_col] = name
    # batch_col == "batch" so no rename needed

    adata.obs["batch"] = adata.obs["batch"].astype(str).astype("category")

    # Label column
    if label_col:
        if label_col != "label":
            adata.obs["label"] = adata.obs[label_col]
            del adata.obs[label_col]
        if unknown_label != "unknown":
            adata.obs["label"].replace({unknown_label: "unknown"}, inplace=True)
        adata.obs["label"] = adata.obs["label"].astype(str).fillna("unknown")
        adata.obs["label"] = adata.obs["label"].map(to_Florent_case)
    else:
        adata.obs["label"] = "unknown"
    adata.obs["label"] = adata.obs["label"].astype("category")

    # Sample column
    adata.obs["sample"] = name
    adata.obs["sample"] = adata.obs["sample"].astype("category")

    # Suffix batch with sample to avoid overlap
    adata.obs["batch"] = (adata.obs["batch"].astype(str) + "_" + adata.obs["sample"].astype(str)).astype("category")

    out_path = os.path.join(unify_dir, f"{name}_unified.h5ad")
    adata.write_h5ad(out_path)
    unified_paths[name] = out_path
    print(f"  {name}: {adata.n_obs} cells, {adata.n_vars} genes")

print("UNIFY:ADATA_UNIFY — done.")


Cell 17 | markdown
### Step 3.2. UNIFY: UPSET_GENES

In [ ]:
# Cell 18 | code
import warnings
import upsetplot

upset_genes_dir = os.path.join(unify_dir, "upset_genes")
os.makedirs(upset_genes_dir, exist_ok=True)

adata_genes = {
    name: ad.read_h5ad(path, backed="r").var.index.unique().tolist()
    for name, path in unified_paths.items()
}

if len(adata_genes) >= 2:
    plot_data = upsetplot.from_contents(adata_genes)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        upsetplot.plot(plot_data, sort_by="cardinality", show_counts=True, min_subset_size=10)
    plot_path = os.path.join(upset_genes_dir, "upset_genes.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {plot_path}")
else:
    print("  Skipped: need at least 2 samples for UpSet plot")

print("UNIFY:UPSET_GENES — done.")


Cell 19 | markdown
### Step 4. COMBINE: ADATA_MERGE

In [ ]:
# Cell 20 | code
import scipy
import pickle
from collections import defaultdict

combine_dir = "$path/4.combine"
os.makedirs(combine_dir, exist_ok=True)

prefix = "merged"
force_obs_cols = ""  # from nextflow.config default

# Load all unified h5ads (no base adata in this run)
adatas = [sc.read_h5ad(p) for p in sorted(unified_paths.values())]
genes = [adata.var_names for adata in adatas]

# Determine obs column intersection (+ any forced columns)
obs_col_intersection = set(adatas[0].obs.columns).intersection(
    *[adata.obs.columns for adata in adatas[1:]]
)
if force_obs_cols:
    obs_col_intersection = obs_col_intersection.union(force_obs_cols.split(","))
obs_col_intersection = sorted(obs_col_intersection)

# Collect column dtype info across all adatas
column_dtypes = defaultdict(set)
for adata in adatas:
    for dtype in ["object", "category", "number"]:
        for col in adata.obs.select_dtypes(include=dtype).columns:
            column_dtypes[col].add(dtype)

column_defaults = {}
for col, dtypes in column_dtypes.items():
    if len(dtypes) > 1:
        for adata in adatas:
            if col in adata.obs.columns:
                adata.obs[col] = adata.obs[col].astype("object")
        column_defaults[col] = "unknown"
    else:
        column_defaults[col] = np.nan if dtypes.copy().pop() == "number" else "unknown"

# Align obs columns
for adata in adatas:
    for col in set(obs_col_intersection).difference(adata.obs.columns):
        adata.obs[col] = column_defaults[col]
    adata.obs = adata.obs[obs_col_intersection]
    for col in obs_col_intersection:
        if column_dtypes[col] == {"number"}:
            continue
        adata.obs[col] = adata.obs[col].astype(str).astype("category")

# Outer concat
adata_outer = ad.concat(adatas, join="outer")
adata_outer.X = csr_matrix(adata_outer.X)
adata_outer.obs = adata_outer.obs.reindex(sorted(adata_outer.obs.columns), axis=1)

# Inner (gene intersection)
gene_intersection = set(genes[0]).intersection(*genes[1:])
intersection_mask = adata_outer.var_names.to_series(name="intersection").map(lambda x: x in gene_intersection)
adata_inner = adata_outer[:, intersection_mask].copy()

# Save
adata_outer.write_h5ad(os.path.join(combine_dir, f"{prefix}_outer.h5ad"))
adata_inner.write_h5ad(os.path.join(combine_dir, f"{prefix}_inner.h5ad"))
# No base adata → integrate = inner (symlink equivalent: just copy path)
adata_inner.write_h5ad(os.path.join(combine_dir, f"{prefix}_integrate.h5ad"))
intersection_mask.to_pickle(os.path.join(combine_dir, "gene_intersection.pkl"))

print(f"  Outer  : {adata_outer.n_obs} cells x {adata_outer.n_vars} genes")
print(f"  Inner  : {adata_inner.n_obs} cells x {adata_inner.n_vars} genes (shared genes)")
print(f"  Output : {combine_dir}")
print("COMBINE:ADATA_MERGE — done.")


Cell 21 | markdown
## Step 5 — Cell Type Annotation with CellTypist

Annotate cells using the **Immune All Low** atlas model (`Immune_All_Low.pkl`) before HVG selection.

**Input:** `merged_integrate.h5ad` from Step 4 combine (`4.combine/merged_integrate.h5ad`).  
**Model:** `$path/scripts/single_cell_math/test_data/atlas/Immune_All_Low.pkl`  
**Output:** `merged_integrate_celltypist.h5ad` saved under `5.pre_hvg_annotation/` with `celltypist:*` columns added to `adata.obs`.

> **Note on gene names:** CellTypist requires gene symbols as `var_names`. If your `var_names` are  
> Ensembl IDs, uncomment the `adata_ct.var_names = ...` line and set `symbol_col` to the correct column.

In [ ]:
# Cell 22 | code
import os
import pandas as pd
import scanpy as sc
import celltypist
from celltypist import models as ct_models
import mygene

# ── Paths ──────────────────────────────────────────────────────────────────────
input_h5ad  = "$path/4.combine/merged_integrate.h5ad"
model_path  = "$path/scripts/single_cell_math/test_data/atlas/Immune_All_Low.pkl" #### please change to your local model path
annot_dir   = "$path/5.pre_hvg_annotation"
out_h5ad    = os.path.join(annot_dir, "merged_celltypist_immune_low.h5ad")
out_obs_pkl = os.path.join(annot_dir, "celltypist_obs.pkl")
os.makedirs(annot_dir, exist_ok=True)

# ── Load AnnData ───────────────────────────────────────────────────────────────
adata = sc.read_h5ad(input_h5ad)
print(f"Loaded: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"var_names (first 5): {list(adata.var_names[:5])}")

# ── Prepare a copy for CellTypist (requires normalised log counts) ─────────────
adata_ct = adata.copy()
sc.pp.normalize_per_cell(adata_ct, counts_per_cell_after=1e4)
sc.pp.log1p(adata_ct)

# If var_names are Ensembl IDs, switch to gene symbols before annotating.
# Uncomment and set symbol_col to the relevant column in adata.var (e.g. "gene_symbols").
# symbol_col = "gene_symbols"
# adata_ct.var_names = adata_ct.var[symbol_col]

# ── Load local model ───────────────────────────────────────────────────────────
model_name = os.path.basename(model_path)[:-4]   # "Mouse_Whole_Brain"
model_obj  = ct_models.Model.load(model_path)
print(f"Model loaded: {model_name}")
print(f"Model cell types: {len(model_obj.cell_types)}")

# ── Run annotation ─────────────────────────────────────────────────────────────
# Make sure the annotation object uses gene symbols, not Ensembl IDs.
# CellTypist requires `var_names` to match model features.
symbol_col = None
for c in ("gene_symbols", "symbol", "gene_symbol", "Symbol", "GeneSymbol"):
    if c in adata_ct.var.columns:
        symbol_col = c
        break

if symbol_col is not None:
    adata_ct.var_names = adata_ct.var[symbol_col].astype(str)
else:
    adata_ct.var_names = adata_ct.var_names.astype(str)

# Keep only genes present in the loaded CellTypist model.
model_features = set(model_obj.features)
overlap_mask = [g in model_features for g in adata_ct.var_names]
if not any(overlap_mask):
    # CellTypist features are stored as gene symbols; many AnnData objects use
    # a different case or a symbol column. Try a case-insensitive alignment first.
    model_features_uc = {str(g).upper() for g in model_obj.features}
    var_names_uc = adata_ct.var_names.astype(str).str.upper()

    overlap_mask = [g in model_features_uc for g in var_names_uc]
    if any(overlap_mask):
        adata_ct.var_names = var_names_uc
        adata_ct = adata_ct[:, overlap_mask].copy()
    else:
        # If a dedicated symbol column exists, use it explicitly.
        if symbol_col is not None:
            adata_ct.var_names = adata_ct.var[symbol_col].astype(str).str.upper()
            overlap_mask = [g in model_features_uc for g in adata_ct.var_names]
            if any(overlap_mask):
                adata_ct = adata_ct[:, overlap_mask].copy()
            else:
                raise ValueError(
                    "No features overlap with the model after symbol remapping. "
                    "Please provide gene symbols in `adata_ct.var_names` or in a "
                    "`gene_symbols`/`symbol` column that matches the CellTypist model features."
                )
        else:
            # If the data are still using Ensembl IDs, try an Ensembl -> gene-symbol lookup.
            # This is the common case when `adata.var_names` are `ENSG...` IDs.
            if all(str(g).startswith("ENSG") for g in adata_ct.var_names[: min(100, len(adata_ct.var_names))]):
                try:
                    mg = mygene.MyGeneInfo()

                    mapping = mg.querymany(
                        list(adata_ct.var_names),
                        scopes="ensembl.gene",
                        fields="symbol",
                        species="human",
                        returnall=False,
                        verbose=False,
                    )

                    symbol_map = {}
                    for item in mapping:
                        q = item.get("query")
                        if q is None:
                            continue
                        symbol = item.get("symbol")
                        if symbol:
                            symbol_map[q] = symbol

                    if symbol_map:
                        adata_ct.var_names = pd.Index(
                            [symbol_map.get(str(g), str(g)) for g in adata_ct.var_names]
                        )
                        model_features_uc = {str(g).upper() for g in model_obj.features}
                        var_names_uc = adata_ct.var_names.astype(str).str.upper()

                        overlap_mask = [g in model_features_uc for g in var_names_uc]
                        if any(overlap_mask):
                            adata_ct = adata_ct[:, overlap_mask].copy()
                        else:
                            raise ValueError(
                                "No features overlap with the model after Ensembl-to-symbol mapping. "
                                "Please provide gene symbols in `adata_ct.var_names` or in a "
                                "`gene_symbols`/`symbol` column that matches the CellTypist model features."
                            )
                    else:
                        raise ValueError(
                            "No features overlap with the model. "
                            "Your `adata_ct.var_names` appear to be Ensembl IDs, but no gene-symbol "
                            "mapping was found. Please provide gene symbols in `adata_ct.var_names` "
                            "or in a `gene_symbols`/`symbol` column."
                        )
                except Exception:
                    raise ValueError(
                        "No features overlap with the model. "
                        "Your `adata_ct.var_names` do not match the CellTypist gene symbols "
                        "(case-insensitive overlap was also unsuccessful, and Ensembl-to-symbol "
                        "mapping was not available). Please provide gene symbols in "
                        "`adata_ct.var_names` or in a `gene_symbols`/`symbol` column."
                    ) from None
            else:
                raise ValueError(
                    "No features overlap with the model. "
                    "Your `adata_ct.var_names` do not match the CellTypist gene symbols "
                    "(case-insensitive overlap was also unsuccessful). "
                    "Please provide gene symbols in `adata_ct.var_names` or in a "
                    "`gene_symbols`/`symbol` column."
                )

# Rebuild the overlap mask against the current adata_ct shape (5782 vars),
# instead of reusing the stale mask that was computed before symbol remapping.
current_var_names = adata_ct.var_names.astype(str)
overlap_mask = [g in model_features for g in current_var_names]

if not any(overlap_mask):
    model_features_uc = {str(g).upper() for g in model_obj.features}
    overlap_mask = [str(g).upper() in model_features_uc for g in current_var_names]

if not any(overlap_mask):
    raise ValueError(
        "No features overlap with the model after current name normalization."
    )

adata_ct = adata_ct[:, overlap_mask].copy()

# Run annotation on the overlap-only object.
predictions = celltypist.annotate(
    adata_ct,
    model=model_obj,
    majority_voting=True,
)
predictions_adata = predictions.to_adata()

df_ct = predictions_adata.obs.loc[
    adata.obs.index, ["predicted_labels", "conf_score"]
].copy()
df_ct.columns = [f"celltypist:{model_name}", f"celltypist:{model_name}:conf"]

# ── Merge results back into original adata ─────────────────────────────────────
adata.obs = pd.concat([adata.obs, df_ct], axis=1)

# ── Save ───────────────────────────────────────────────────────────────────────
df_ct.to_pickle(out_obs_pkl)
adata.write_h5ad(out_h5ad)

print(f"\nAnnotation complete.")
print(f"  Unique cell types assigned : {adata.obs[f'celltypist:{model_name}'].nunique()}")
print(f"  Output h5ad  : {out_h5ad}")
print(f"  Output obs   : {out_obs_pkl}")